# 阶段四：真实 Reward 预检与性能基线

本 Notebook 只评估少量人工表达式，不创建 `GFNTrainer`、不更新参数、不写检查点，也不修改任何数据文件。

运行要求：

1. 先重启内核，确保内存基线未被旧对象污染；
2. 从上到下顺序运行；
3. 保留全部输出供人工审阅，但不要将运行输出提交 Git；
4. 本 Notebook 只报告实测耗时和内存，不自行设定性能通过阈值，也不会自动进入训练。

## 1. 环境、导入与项目路径

In [ ]:
import gc
import os
import platform
import sys
import threading
import warnings
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import psutil
from IPython.display import display

working_dir = Path.cwd().resolve()
project_candidates = (working_dir, *working_dir.parents)
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / 'factor_gfn').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('无法从当前目录向上找到 factor_gfn 项目根目录')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from factor_gfn.barra import STYLE_NAMES
from factor_gfn.evaluator.cross_section import IndustryNeutralizationWarning
from factor_gfn.gfn import (
    DEFAULT_REAL_REWARD_CONFIG,
    RealRewardDataPaths,
    RealRewardProvider,
    build_real_reward_data_context,
)
from factor_gfn.grammar import Expression, get_action_id

environment = {
    'project_root': str(PROJECT_ROOT),
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'psutil': psutil.__version__,
    'pid': os.getpid(),
}
display(pd.DataFrame([environment]))

## 2. 进程 RSS 峰值采样工具

这里测量的是当前 Jupyter 内核进程的 RSS，而不是 `tracemalloc` 的 Python 堆。阶段内峰值通过后台线程定期采样，因此同时报告采样间隔、阶段前后 RSS 和峰值增量。

In [ ]:
RSS_SAMPLE_INTERVAL_SECONDS = 0.02
PROCESS = psutil.Process(os.getpid())


def bytes_to_mib(value):
    return float(value) / (1024.0 ** 2)


class RssPeakSampler:
    def __init__(self, interval_seconds=RSS_SAMPLE_INTERVAL_SECONDS):
        self.interval_seconds = float(interval_seconds)
        self.rss_before = 0
        self.rss_peak = 0
        self._stop = threading.Event()
        self._thread = None

    def _sample(self):
        while not self._stop.is_set():
            self.rss_peak = max(self.rss_peak, PROCESS.memory_info().rss)
            self._stop.wait(self.interval_seconds)

    def __enter__(self):
        self.rss_before = PROCESS.memory_info().rss
        self.rss_peak = self.rss_before
        self._thread = threading.Thread(target=self._sample, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.rss_peak = max(self.rss_peak, PROCESS.memory_info().rss)
        self._stop.set()
        self._thread.join()


def measure_phase(label, operation):
    gc.collect()
    with RssPeakSampler() as sampler:
        started = perf_counter()
        value = operation()
        wall_seconds = perf_counter() - started
    rss_after = PROCESS.memory_info().rss
    profile = {
        'phase': label,
        'wall_seconds': wall_seconds,
        'rss_before_mib': bytes_to_mib(sampler.rss_before),
        'rss_after_mib': bytes_to_mib(rss_after),
        'rss_peak_mib': bytes_to_mib(sampler.rss_peak),
        'rss_peak_delta_mib': bytes_to_mib(
            max(0, sampler.rss_peak - sampler.rss_before)
        ),
        'sample_interval_seconds': sampler.interval_seconds,
    }
    return value, profile

## 3. 输入文件只读预检

In [ ]:
data_paths = RealRewardDataPaths()
barra_paths = data_paths.barra_paths
required_paths = [
    data_paths.tensor_path,
    data_paths.universe_mask_path,
    data_paths.date_list_path,
    data_paths.stock_list_path,
    data_paths.processed_metadata_path,
    data_paths.industry_path,
    data_paths.industry_metadata_path,
    barra_paths.metadata_path,
    barra_paths.market_return_path,
    *[barra_paths.exposure_path(name) for name in STYLE_NAMES],
]
input_rows = [
    {
        'path': str(path),
        'exists': path.is_file(),
        'size_mib': bytes_to_mib(path.stat().st_size) if path.is_file() else np.nan,
    }
    for path in required_paths
]
input_frame = pd.DataFrame(input_rows)
display(input_frame)
missing_paths = input_frame.loc[~input_frame['exists'], 'path'].tolist()
if missing_paths:
    raise FileNotFoundError(f'真实 Reward 输入缺失：{missing_paths}')

## 4. 构建真实数据上下文并核对标签边界

该单元可能读取较大的行业长表并构造五条 Barra LS。它不会计算候选表达式。

In [ ]:
context, context_profile = measure_phase(
    'build_real_reward_data_context',
    build_real_reward_data_context,
)

manifest = context.manifest
calendar = manifest['calendar']
train_end = np.datetime64(context.config.train_end, 'D')
entry_lag = context.config.evaluation.entry_lag
horizon = context.config.evaluation.horizon
last_rebalance_row = int(context.rebalance_indices[-1])
last_exit_row = last_rebalance_row + entry_lag + horizon
if last_exit_row >= context.evaluation_dates.size:
    raise AssertionError('最后一个调仓日无法在训练评价轴内取得完整退出价格')
last_exit_date = context.evaluation_dates[last_exit_row]

assert context.history_dates[-1] <= train_end
assert context.evaluation_dates[-1] <= train_end
assert last_exit_date <= train_end
assert manifest['industry']['universe_missing_count'] == 0
assert set(context.barra_long_short) == set(STYLE_NAMES)

barra_reference_rows = []
for style_name in STYLE_NAMES:
    values = context.barra_long_short[style_name].long_short_return[
        context.rebalance_indices
    ]
    valid_periods = int(np.isfinite(values).sum())
    barra_reference_rows.append(
        {
            'style': style_name,
            'valid_periods': valid_periods,
            'total_rebalance_periods': int(context.rebalance_indices.size),
            'all_calendar_periods_valid': bool(
                valid_periods == context.rebalance_indices.size
            ),
        }
    )
barra_reference_frame = pd.DataFrame(barra_reference_rows)

context_summary = {
    'context_fingerprint': context.fingerprint,
    'history_shape': tuple(context.factor_tensor.shape),
    'evaluation_shape': tuple(context.forward_returns.shape),
    'actual_train_start': str(context.evaluation_dates[0]),
    'actual_train_end': str(context.evaluation_dates[-1]),
    'first_rebalance_date': calendar['first_rebalance_date'],
    'last_rebalance_date': calendar['last_rebalance_date'],
    'rebalance_periods': calendar['rebalance_periods'],
    'last_label_exit_date': str(last_exit_date),
    'history_excludes_validation': bool(context.history_dates[-1] <= train_end),
    'universe_industry_missing_count': manifest['industry']['universe_missing_count'],
}
display(pd.DataFrame([context_summary]))
display(barra_reference_frame)
display(pd.DataFrame([context_profile]))

## 5. 构建真实 RewardProvider

此处会初始化 `FactorInterpreter`。当前实现会将训练历史六特征张量复制为 `float64`，所以必须单独记录该阶段的时间和 RSS 峰值。

In [ ]:
provider, provider_profile = measure_phase(
    'RealRewardProvider.__init__',
    lambda: RealRewardProvider(
        context,
        DEFAULT_REAL_REWARD_CONFIG,
    ),
)
provider_manifest = provider.manifest()
assert provider_manifest['industry_neutralization'] is True
assert provider_manifest['calendar']['periods'] == context.rebalance_indices.size
assert provider.reward_config.candidate_industry_neutralization is True
display(pd.DataFrame([provider_profile]))
display(pd.DataFrame([provider.cache_info()]))

## 6. 构造五个人工表达式

覆盖叶子、单输入时序、二元组合、另一种时序输入以及截面算子。这里只构造表达式，不评价。

In [ ]:
close_id = get_action_id('close')
volume_id = get_action_id('volume')
delta_20_id = get_action_id('ts_delta', 20)
delay_20_id = get_action_id('ts_delay', 20)
mean_20_id = get_action_id('ts_mean', 20)
div_id = get_action_id('div')
cs_rank_id = get_action_id('cs_rank')

expressions = {
    'close': Expression.from_prefix([close_id]),
    'ts_delta_close_20': Expression.from_prefix([delta_20_id, close_id]),
    'delta_over_delay_20': Expression.from_prefix(
        [div_id, delta_20_id, close_id, delay_20_id, close_id]
    ),
    'ts_mean_volume_20': Expression.from_prefix([mean_20_id, volume_id]),
    'cs_rank_close': Expression.from_prefix([cs_rank_id, close_id]),
}
expression_frame = pd.DataFrame(
    [
        {
            'name': name,
            'formula': expression.to_formula(),
            'structural_hash': expression.structural_hash(),
            'node_count': expression.stats.node_count,
            'depth': expression.stats.depth,
            'prefix_token_ids': list(expression.to_prefix()),
        }
        for name, expression in expressions.items()
    ]
)
assert expression_frame['structural_hash'].is_unique
display(expression_frame)

## 7. 首次真实评价：耗时、峰值内存与 Reward 拆解

无效指标会由 Provider 返回明确拒绝原因；形状、解释器或数据合同异常仍应直接中止。行业中性化警告会被收集并显示，不能静默忽略。

In [ ]:
def evaluate_first_pass(name, expression):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', IndustryNeutralizationWarning)
        assignment, profile = measure_phase(
            f'evaluate:{name}',
            lambda: provider.evaluate(expression),
        )
    industry_warnings = [
        item
        for item in caught
        if issubclass(item.category, IndustryNeutralizationWarning)
    ]
    metadata = assignment.metadata
    result = metadata['reward_result']
    row = {
        'name': name,
        'formula': metadata['formula'],
        'structural_hash': metadata['expression_hash'],
        'valid': assignment.valid,
        'rejection_reason': assignment.rejection_reason,
        'provider_cache_hit': metadata['provider_cache_hit'],
        'reward_evaluator_cache_hit': metadata['reward_evaluator_cache_hit'],
        'factor_seconds': metadata['factor_seconds'],
        'reward_seconds': metadata['reward_seconds'],
        'total_wall_seconds': profile['wall_seconds'],
        'rss_peak_mib': profile['rss_peak_mib'],
        'rss_peak_delta_mib': profile['rss_peak_delta_mib'],
        'finite_universe_count': metadata['finite_universe_count'],
        'universe_count': metadata['universe_count'],
        'finite_universe_coverage': metadata['finite_universe_coverage'],
        'train_ic': result['train_ic'],
        'ic_valid_periods': result['ic_valid_periods'],
        'train_long_ir': result['train_long_ir'],
        'long_ir_valid_periods': result['long_ir_valid_periods'],
        'barra_ts_corr': result['barra_ts_corr'],
        'dominant_barra_factor': result['dominant_barra_factor'],
        'dominant_barra_correlation': result['dominant_barra_correlation'],
        'raw_reward': result['raw_reward'],
        'reward': result['reward'],
        'log_reward': result['log_reward'],
        'floor_applied': result['floor_applied'],
        'long_direction': result['long_direction'],
        'industry_neutralized': result['industry_neutralized'],
        'neutralization_skipped_dates': result['neutralization_skipped_dates'],
        'neutralization_skipped_rate': result['neutralization_skipped_rate'],
        'industry_warning_count': len(industry_warnings),
        'industry_warnings': ' | '.join(
            str(item.message) for item in industry_warnings
        ),
    }
    for style_name in STYLE_NAMES:
        row[f'barra_corr_{style_name}'] = result['barra_correlations'][style_name]
        row[f'barra_periods_{style_name}'] = result['barra_valid_periods'][style_name]
    return row


first_pass_rows = [
    evaluate_first_pass(name, expression)
    for name, expression in expressions.items()
]
first_pass_frame = pd.DataFrame(first_pass_rows)
assert not first_pass_frame['provider_cache_hit'].any()
assert provider.interpreter_evaluation_count == len(expressions)
assert len(provider.evaluation_records) == len(expressions)
display(
    first_pass_frame[
        [
            'name', 'valid', 'rejection_reason',
            'factor_seconds', 'reward_seconds', 'total_wall_seconds',
            'rss_peak_delta_mib', 'finite_universe_coverage',
            'train_ic', 'ic_valid_periods', 'train_long_ir',
            'long_ir_valid_periods', 'barra_ts_corr',
            'dominant_barra_factor', 'dominant_barra_correlation',
            'raw_reward', 'reward', 'industry_neutralized',
            'neutralization_skipped_dates', 'neutralization_skipped_rate',
            'industry_warning_count',
        ]
    ]
)

In [ ]:
barra_columns = ['name']
for style_name in STYLE_NAMES:
    barra_columns.extend(
        [f'barra_corr_{style_name}', f'barra_periods_{style_name}']
    )
display(first_pass_frame[barra_columns])

## 8. 重复评价：验证 Provider 缓存

缓存返回的 metadata 保留首次评价的 `factor_seconds` 和 `reward_seconds`。下面只把本次调用的 wall time 作为缓存耗时，并验证解释器执行数及评价记录数均不增加。

In [ ]:
cache_rows = []
for name, expression in expressions.items():
    interpreter_before = provider.interpreter_evaluation_count
    records_before = len(provider.evaluation_records)
    hits_before = provider.cache_hit_count
    assignment, profile = measure_phase(
        f'cache_hit:{name}',
        lambda expression=expression: provider.evaluate(expression),
    )
    cache_rows.append(
        {
            'name': name,
            'valid': assignment.valid,
            'provider_cache_hit': assignment.metadata['provider_cache_hit'],
            'cache_wall_seconds': profile['wall_seconds'],
            'rss_peak_delta_mib': profile['rss_peak_delta_mib'],
            'interpreter_evaluation_delta': (
                provider.interpreter_evaluation_count - interpreter_before
            ),
            'evaluation_record_delta': (
                len(provider.evaluation_records) - records_before
            ),
            'cache_hit_delta': provider.cache_hit_count - hits_before,
        }
    )
cache_frame = pd.DataFrame(cache_rows)
assert cache_frame['provider_cache_hit'].all()
assert (cache_frame['interpreter_evaluation_delta'] == 0).all()
assert (cache_frame['evaluation_record_delta'] == 0).all()
assert (cache_frame['cache_hit_delta'] == 1).all()
display(cache_frame)
display(pd.DataFrame([provider.cache_info()]))

## 9. 进入训练前合同汇总

自动检查只覆盖数据与调用合同。`若干个有效候选`的具体数量以及时间/内存是否可接受仍由人工审阅决定；本 Notebook 不会据此自动开始训练或优化。

In [ ]:
valid_frame = first_pass_frame[first_pass_frame['valid']]
valid_reward_count = int(valid_frame.shape[0])
reference_min_periods = int(barra_reference_frame['valid_periods'].min())
valid_correlations_complete = bool(
    valid_reward_count > 0
    and np.isfinite(
        valid_frame[[f'barra_corr_{name}' for name in STYLE_NAMES]].to_numpy(
            dtype=float
        )
    ).all()
)
valid_common_periods_sufficient = bool(
    valid_reward_count > 0
    and (
        valid_frame[[f'barra_periods_{name}' for name in STYLE_NAMES]]
        .to_numpy(dtype=int)
        >= DEFAULT_REAL_REWARD_CONFIG.barra_min_common_periods
    ).all()
)

automatic_checks = [
    {
        'criterion': '存在至少一个有效 Reward，完整链路可达',
        'status': valid_reward_count > 0,
        'detail': f'有效候选 {valid_reward_count}/{len(expressions)}；若干个的门槛人工确认',
    },
    {
        'criterion': '行业中性化启用且无跳过警告',
        'status': bool(
            first_pass_frame['industry_neutralized'].all()
            and (first_pass_frame['industry_warning_count'] == 0).all()
            and (first_pass_frame['neutralization_skipped_rate'] == 0.0).all()
            and manifest['industry']['universe_missing_count'] == 0
        ),
        'detail': (
            f"warnings={int(first_pass_frame['industry_warning_count'].sum())}, "
            f"persisted_skipped={sum(len(value) for value in first_pass_frame['neutralization_skipped_dates'])}, "
            f"universe_missing={manifest['industry']['universe_missing_count']}"
        ),
    },
    {
        'criterion': '五条 Barra LS 基准均有充足有效期',
        'status': bool(
            (barra_reference_frame['valid_periods'] >= 
             DEFAULT_REAL_REWARD_CONFIG.barra_min_common_periods).all()
        ),
        'detail': f'五条基准最少有效期={reference_min_periods}',
    },
    {
        'criterion': '有效候选五项 Barra 相关均有限且共同期数达标',
        'status': valid_correlations_complete and valid_common_periods_sufficient,
        'detail': f'最低共同期门槛={DEFAULT_REAL_REWARD_CONFIG.barra_min_common_periods}',
    },
    {
        'criterion': '无标签越界或验证集特征泄露',
        'status': bool(
            context.history_dates[-1] <= train_end
            and context.evaluation_dates[-1] <= train_end
            and last_exit_date <= train_end
        ),
        'detail': (
            f'history_end={context.history_dates[-1]}, '
            f'last_label_exit={last_exit_date}, frozen_end={train_end}'
        ),
    },
    {
        'criterion': '重复评价全部命中解释前缓存',
        'status': bool(
            cache_frame['provider_cache_hit'].all()
            and (cache_frame['interpreter_evaluation_delta'] == 0).all()
        ),
        'detail': f"hits={provider.cache_hit_count}, requests={provider.request_count}",
    },
]
automatic_check_frame = pd.DataFrame(automatic_checks)
display(automatic_check_frame)

phase_profiles = pd.DataFrame([context_profile, provider_profile])
display(phase_profiles)
display(
    first_pass_frame[
        [
            'name', 'factor_seconds', 'reward_seconds', 'total_wall_seconds',
            'rss_peak_mib', 'rss_peak_delta_mib', 'valid', 'reward',
        ]
    ]
)

automatic_contracts_ok = bool(automatic_check_frame['status'].all())
print(f'自动合同检查通过：{automatic_contracts_ok}')
print(f'有效 Reward 候选数：{valid_reward_count}/{len(expressions)}')
print('请人工审阅耗时、RSS 峰值和有效候选数量；本 Notebook 不宣布进入训练。')

## 10. 手工运行后的回传内容

请保留并回传以下输出：

- 上下文与 Provider 两阶段的性能表；
- 首次表达式评价主表；
- 五项 Barra 相关及共同期数表；
- 缓存复测表；
- 进入训练前合同汇总。

如果耗时或内存偏高，先依据这些分阶段数据定位瓶颈，再单独决定是否优化。